In [ ]:
import sys
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

sys.path.append(str(Path.cwd().parent))
from src.database import SalesDatabase

db = SalesDatabase()
db.connect()

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [ ]:
PROJECT_ROOT = Path.cwd().parent
SQL_DIR = PROJECT_ROOT / 'sql'
RESULTADOS_DIR = PROJECT_ROOT / 'data' / 'processed' / 'resultados'
RESULTADOS_DIR.mkdir(parents=True, exist_ok=True)

def executar_sql(nome_arquivo):
    caminho = SQL_DIR / nome_arquivo
    query = caminho.read_text(encoding='utf-8')
    return db.execute_query(query)

def salvar_resultado(df, nome):
    caminho = RESULTADOS_DIR / f'{nome}.csv'
    df.to_csv(caminho, index=False)
    print(f'Resultado salvo em: {caminho} ({len(df)} linhas)')
    return caminho

def ic_normal(diferenca, se):
    """Intervalo de confiança 95% (aproximação normal)."""
    return round(diferenca - 1.96 * se, 4), round(diferenca + 1.96 * se, 4)

def ic_fisher(r, n):
    """Intervalo de confiança 95% para correlação via transformação z de Fisher."""
    z = np.arctanh(r)
    se = 1 / np.sqrt(n - 3)
    return round(np.tanh(z - 1.96 * se), 4), round(np.tanh(z + 1.96 * se), 4)

def cohen_d(g1, g2):
    """Tamanho de efeito (d de Cohen) entre dois grupos."""
    m1, m2 = g1.mean(), g2.mean()
    s1, s2 = g1.std(ddof=1), g2.std(ddof=1)
    n1, n2 = len(g1), len(g2)
    pooled = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
    return (m1 - m2) / pooled

In [ ]:
# Dataset de avaliações vinculado ao atraso de entrega
# (apenas pedidos entregues, com data real e estimada de entrega)

query_avaliacoes = """
SELECT  o.order_id,
        r.review_score,
        CAST(julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) AS INTEGER) AS atraso_dias,
        CAST(julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp) AS INTEGER) AS tempo_entrega_dias,
        COALESCE(p.payment_value, 0) AS valor_pedido
FROM orders o
JOIN order_reviews r ON o.order_id = r.order_id
LEFT JOIN order_payments p ON o.order_id = p.order_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL;
"""

df_avaliacoes = db.execute_query(query_avaliacoes)
df_avaliacoes['no_prazo'] = df_avaliacoes['atraso_dias'] <= 0
df_avaliacoes['faixa_atraso'] = pd.cut(
    df_avaliacoes['atraso_dias'],
    bins=[-np.inf, 0, 3, 10, np.inf],
    labels=['No prazo', 'Atraso 1-3 dias', 'Atraso 4-10 dias', 'Atraso > 10 dias']
)

print(f'Pedidos avaliados: {len(df_avaliacoes)}')
print(f'Atraso médio: {df_avaliacoes["atraso_dias"].mean():.1f} dias')
print(df_avaliacoes['faixa_atraso'].value_counts().sort_index().to_string())

In [ ]:
# Datasets auxiliares: pedidos diários e itens (preço x frete)

query_diario = """
SELECT  date(order_purchase_timestamp) AS dia,
        COUNT(*) AS pedidos
FROM orders
WHERE order_status = 'delivered'
GROUP BY date(order_purchase_timestamp)
ORDER BY dia;
"""

query_itens = """
SELECT  price,
        freight_value
FROM order_items;
"""

df_pedidos_diarios = db.execute_query(query_diario)
df_pedidos_diarios['mes'] = df_pedidos_diarios['dia'].str[5:7]
df_pedidos_diarios['safra'] = df_pedidos_diarios['mes'].isin(['11', '12'])

df_itens = db.execute_query(query_itens)

print(f'Dias com pedidos: {len(df_pedidos_diarios)}')
print(f'Itens: {len(df_itens)}')

In [ ]:
# Análise descritiva da avaliação

print('Distribuição das notas:')
print(df_avaliacoes['review_score'].value_counts().sort_index().to_string())

print('\nNota média por faixa de atraso:')
resumo_faixa = df_avaliacoes.groupby('faixa_atraso', observed=True).agg(
    pedidos=('review_score', 'count'),
    nota_media=('review_score', 'mean'),
    nota_mediana=('review_score', 'median')
).round(2)
print(resumo_faixa.to_string())
salvar_resultado(resumo_faixa.reset_index(), '19_nota_media_por_faixa_atraso')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
df_avaliacoes['review_score'].value_counts().sort_index().plot(kind='bar', ax=axes[0])
axes[0].set_title('Distribuição das Notas de Avaliação')
axes[0].set_xlabel('Nota')
axes[0].set_ylabel('Quantidade')

sns.boxplot(data=df_avaliacoes, x='faixa_atraso', y='review_score', ax=axes[1])
axes[1].set_title('Nota por Faixa de Atraso')
axes[1].set_xlabel('Faixa de atraso')
axes[1].set_ylabel('Nota')
plt.tight_layout()
plt.show()

In [ ]:
# TESTE A - Entrega atrasada afeta a avaliação do cliente?

no_prazo = df_avaliacoes.loc[df_avaliacoes['no_prazo'], 'review_score']
com_atraso = df_avaliacoes.loc[~df_avaliacoes['no_prazo'], 'review_score']

n1, n2 = len(no_prazo), len(com_atraso)
m1, m2 = no_prazo.mean(), com_atraso.mean()
s1, s2 = no_prazo.std(ddof=1), com_atraso.std(ddof=1)

diferenca = m1 - m2
se = np.sqrt(s1**2 / n1 + s2**2 / n2)

# Teste t de Welch (paramétrico, robusto a amostras grandes)
t_stat, p_t = stats.ttest_ind(no_prazo, com_atraso, equal_var=False)

# Mann-Whitney U (recomendado, nota é escala ordinal)
u_stat, p_u = stats.mannwhitneyu(no_prazo, com_atraso, alternative='two-sided')

d = cohen_d(no_prazo, com_atraso)

print('Teste A - No prazo  vs  Com atraso:')
print(f'  n:          {n1:,}  vs  {n2:,}')
print(f'  média nota: {m1:.2f}  vs  {m2:.2f}')
print(f'  diferença:  {diferenca:.2f}  (IC95% {ic_normal(diferenca, se)})')
print(f'  t de Welch: t={t_stat:.2f}, p={p_t:.2e}')
print(f'  Mann-Whitney: U={u_stat:,.0f}, p={p_u:.2e}')
print(f'  d de Cohen: {d:.3f}  ({"efeito pequeno" if abs(d) < 0.2 else "efeito médio" if abs(d) < 0.5 else "efeito grande"})')

In [ ]:
# TESTE B - Correlação entre atraso e avaliação

n = len(df_avaliacoes)

r_pearson, p_pearson = stats.pearsonr(df_avaliacoes['atraso_dias'], df_avaliacoes['review_score'])
r_spearman, p_spearman = stats.spearmanr(df_avaliacoes['atraso_dias'], df_avaliacoes['review_score'])
r_valor, p_valor = stats.pearsonr(df_avaliacoes['valor_pedido'], df_avaliacoes['review_score'])

print('Correlação com a nota de avaliação:')
print(f'  Atraso (dias)   Pearson r={r_pearson:.4f}  (IC95% {ic_fisher(r_pearson, n)})  p={p_pearson:.2e}')
print(f'  Atraso (dias)   Spearman r={r_spearman:.4f}  p={p_spearman:.2e}')
print(f'  Valor do pedido Pearson r={r_valor:.4f}  (IC95% {ic_fisher(r_valor, n)})  p={p_valor:.2e}')
print()
print('Interpretação: quanto maior o atraso, menor a nota.')

In [ ]:
# TESTE C - As faixas de atraso apresentam notas diferentes?
# Kruskal-Wallis (não paramétrico, múltiplos grupos) + post-hoc

faixas = df_avaliacoes['faixa_atraso'].cat.categories
grupos = [df_avaliacoes.loc[df_avaliacoes['faixa_atraso'] == f, 'review_score'].values for f in faixas]

h_stat, p_kw = stats.kruskal(*grupos)
print(f'Kruskal-Wallis: H={h_stat:,.1f}, p={p_kw:.2e}  (n grupos = {len(faixas)})')

# Post-hoc: Mann-Whitney pareado com correção de Bonferroni
n_comparacoes = math.comb(len(faixas), 2)
alpha_bonf = 0.05 / n_comparacoes

print(f'\nPost-hoc (Bonferroni, alpha={alpha_bonf:.4f}):')
for i in range(len(faixas)):
    for j in range(i + 1, len(faixas)):
        g1 = df_avaliacoes.loc[df_avaliacoes['faixa_atraso'] == faixas[i], 'review_score']
        g2 = df_avaliacoes.loc[df_avaliacoes['faixa_atraso'] == faixas[j], 'review_score']
        u, p = stats.mannwhitneyu(g1, g2, alternative='two-sided')
        sig = 'significativo' if p < alpha_bonf else 'não significativo'
        print(f'  {faixas[i]:<18} vs {faixas[j]:<18}  p={p:.2e}  -> {sig}')

In [ ]:
# TESTE D - Correlação entre preço do produto e frete

r_frete, p_frete = stats.pearsonr(df_itens['price'], df_itens['freight_value'])
r_frete_s, p_frete_s = stats.spearmanr(df_itens['price'], df_itens['freight_value'])

print(f'Preço x Frete  Pearson r={r_frete:.4f}  (IC95% {ic_fisher(r_frete, len(df_itens))})  p={p_frete:.2e}')
print(f'Preço x Frete  Spearman r={r_frete_s:.4f}  p={p_frete_s:.2e}')

# Regressão linear simples (numpy) - preço explicando o frete
coef, intercept = np.polyfit(df_itens['price'], df_itens['freight_value'], 1)
print(f'\nRegressão simples: frete = {intercept:.2f} + {coef:.4f} x preço')

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(df_itens['price'], df_itens['freight_value'], alpha=0.15, s=8)
ax.set_xlabel('Preço (R$)')
ax.set_ylabel('Frete (R$)')
ax.set_title('Relação entre Preço e Frete dos Itens')
plt.tight_layout()
plt.show()

In [ ]:
# TESTE E - Sazonalidade: volume na Safra (nov/dez) é maior?

safra = df_pedidos_diarios.loc[df_pedidos_diarios['safra'], 'pedidos']
normal = df_pedidos_diarios.loc[~df_pedidos_diarios['safra'], 'pedidos']

u_saz, p_saz = stats.mannwhitneyu(safra, normal, alternative='greater')

print(f'Pedidos/dia - Safra (nov+dez): média {safra.mean():.0f}, mediana {safra.median():.0f} (n={len(safra)})')
print(f'Pedidos/dia - Demais meses:    média {normal.mean():.0f}, mediana {normal.median():.0f} (n={len(normal)})')
print(f'Mann-Whitney (unilateral: safra > normal): U={u_saz:,.0f}, p={p_saz:.2e}')

print('\nTop 5 dias de maior volume:')
print(df_pedidos_diarios.sort_values('pedidos', ascending=False).head(5).to_string(index=False))

# Pedidos por mês (série)
fig, ax = plt.subplots(figsize=(12, 4))
mensal = df_pedidos_diarios.groupby('mes')['pedidos'].sum()
mensal.plot(kind='bar', ax=ax)
ax.set_title('Pedidos por Mês')
ax.set_xlabel('Mês')
ax.set_ylabel('Pedidos')
plt.tight_layout()
plt.show()

In [ ]:
# Resumo consolidado dos testes de hipótese

conclusoes = pd.DataFrame([
    {'teste': 'A - Atraso x nota (Mann-Whitney)',
     'estatistica': round(float(u_stat), 2), 'p_valor': float(p_u),
     'conclusao': 'Entrega atrasada reduz a nota de forma significativa'},
    {'teste': 'A - Diferença de médias (d de Cohen)',
     'estatistica': round(float(d), 3), 'p_valor': float(p_t),
     'conclusao': f'{abs(d):.2f} = efeito {"grande" if abs(d)>=0.5 else "médio" if abs(d)>=0.2 else "pequeno"}'},
    {'teste': 'B - Correlação atraso x nota (Pearson)',
     'estatistica': round(float(r_pearson), 4), 'p_valor': float(p_pearson),
     'conclusao': 'Correlação negativa moderada entre atraso e satisfação'},
    {'teste': 'C - Faixas de atraso (Kruskal-Wallis)',
     'estatistica': round(float(h_stat), 2), 'p_valor': float(p_kw),
     'conclusao': 'Notas diferem entre todas as faixas de atraso'},
    {'teste': 'D - Preço x frete (Pearson)',
     'estatistica': round(float(r_frete), 4), 'p_valor': float(p_frete),
     'conclusao': 'Correlação moderada entre preço e frete'},
    {'teste': 'E - Sazonalidade Safra (Mann-Whitney)',
     'estatistica': round(float(u_saz), 2), 'p_valor': float(p_saz),
     'conclusao': 'Volume diário maior em novembro/dezembro'}
])

print(conclusoes.to_string(index=False))
salvar_resultado(conclusoes, '20_resumo_testes_hipotese')